In [ ]:
from itertools import product
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd

import os
import matplotlib.pyplot as plt
from glob import glob
import seaborn as sns 
from tqdm.notebook import tqdm
from matplotlib import font_manager
import scipy.stats as stats

font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 12})

In [ ]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [ ]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [ ]:
moldf.shape

In [ ]:
import re

def process_list(data_list):
    """
    Processes a list of strings to perform the following operations:
    1.  Strips trailing newline characters.
    2.  Parses the 'LUCJ' string into three separate elements.
    3.  Converts the final element from a string to a float.
    """
    processed_list = []
    for item in data_list:
        item = item.strip()

        if "LUCJ" in item:
            # Use a regular expression to extract the components
            match = re.search(r'(LUCJ)\(L=(.*?)\)/(.*)', item)
            if match:
                processed_list.extend(match.groups())
            else:
                processed_list.append(item)
        else:
            processed_list.append(item)
    
    # Convert the last element to a float
    # We use a try-except block in case the last element is not a number
    try:
        processed_list[-1] = float(processed_list[-1])
    except (ValueError, IndexError):
        pass # The last element is not a float, so we ignore it

    return processed_list

In [ ]:
postprocessed = []
for i in tqdm(glob("./energies/*txt"),desc='Running'):
    with open(i,'r') as f:
        lines = f.readlines()

    try:  
        moldict = pd.DataFrame.from_dict(dict(zip(["Basis Set","Molecule","Method","L","Injected","Energy"],process_list(lines))),orient='index').T
        moldict.astype({'L':int,"Energy":float})
        postprocessed.append(moldict)
    except ValueError as e:
        print(i)
        print(e)

In [ ]:
LUCJDF=pd.concat(postprocessed).reset_index().drop(columns=['index']).astype({"L":int,"Energy":float})
LUCJDF['Method'] =[f"LUCJ(L={i})" for i in LUCJDF['L']]

In [ ]:
# LUCJDF[LUCJDF['Energy']<-1e3].to_excel("repostprocess.xlsx")

In [ ]:
for basis, layer, mol in product(LUCJDF['Basis Set'].unique(),LUCJDF['L'].unique(),LUCJDF['Molecule'].unique()):
    mask = ((LUCJDF['Basis Set']==basis) &
            (LUCJDF['L']==layer) &
            (LUCJDF['Molecule']==mol) 
           
           )
    
    LUCJDF.loc[mask,'Deviation'] = (LUCJDF.loc[mask,"Energy"] - LUCJDF.loc[mask&(LUCJDF['Injected']=='CCSD'),'Energy'].values)*1e3

In [ ]:
LUCJDF['Deviation'][LUCJDF['Deviation'].abs()>0].describe()

In [ ]:
LUCJDF.loc[LUCJDF['Injected']=='ML_exact','Deviation'].describe()

In [ ]:
LUCJDF.loc[(LUCJDF['Deviation'].abs()>1.6),'Injected']

In [ ]:
sns.barplot(LUCJDF[(LUCJDF['Molecule']=='but-1-yne')&(LUCJDF['Injected']=='CCSD')].sort_values(by=['Basis Set','L']),x='Basis Set',y='Energy',hue='L')

In [ ]:
g = sns.catplot(LUCJDF,
                row='Basis Set',
                col='L',
                x='Molecule',
                y='Deviation',
                hue='Injected',
                kind='bar',
                height=4,
                aspect=0.6,
                )


# Compute global deviation range for consistent shading/scaling
ymin = LUCJDF["Deviation"].min()
ymax = LUCJDF["Deviation"].max()
pad = 0.1 * max(abs(ymin), abs(ymax))

# Apply styling to **all subplots**
for ax in g.axes.flat:
    ax.set_yscale("symlog", linthresh=1e-8)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.axhline(0, color="gray", lw=1, ls="--", alpha=0.8)
    
    # Gray fill region across same y-span
    ax.fill_between(
        np.linspace(-0.5, len(upDF["Method"].unique()) - 0.5, 100),
        -1.6, 1.6,
        color="gray", alpha=0.2
    )

    ax.set_xlim(-0.5, len(upDF["Method"].unique()) - 0.5)

    # Rotate and align x tick labels
    for label in ax.get_xticklabels():
        label.set_rotation(90)
        label.set_horizontalalignment("center")
        label.set_verticalalignment("top")

# Adjust titles and axis labels
g.set_titles(col_template="Layers={col_name}", row_template="{row_name}")
g.set_axis_labels("Method", r"Energy Deviation (m$E_{\mathrm{h}}$)")


# Move legend outside the main plot (right side)
g._legend.set_bbox_to_anchor((1.05, 0.5))
g._legend.set_frame_on(False)
plt.subplots_adjust(right=0.85)  # make space for legend

plt.tight_layout()
plt.show()

In [ ]:
sns.catplot(LUCJDF.sort_values(by=['Basis Set','Molecule','L','Injected']),col='Basis Set',row='L',x='Molecule',y="Energy",hue='Injected',kind='bar')

In [ ]:
for a,b in moldf[['molecule','mol_filename']].values:
    if 'GDB' in b:
        name = b.replace('.xyz','')
        energyDF['Molecule'] = energyDF['Molecule'].replace(name,a)

In [ ]:
energyDF['L'] = len(energyDF)*[np.nan]
energyDF['Injected'] = len(energyDF)*[np.nan]

In [ ]:
upDF = pd.concat([energyDF,LUCJDF]).reset_index().drop(columns=['index'])
upDF['Deviation'] = np.nan

In [ ]:
upDF['Deviation']

In [ ]:
uniqueBasis = energyDF['Basis Set'].unique()
uniqueMol = energyDF['Molecule'].unique()
uniqueInj = energyDF['Injected'].unique()
uniqueLayers = upDF['L'].unique()


In [ ]:
uniqueMol

In [ ]:
upDF['Injected'].value_counts()

In [ ]:
refmethod = "SHCI"
for mol in uniqueMol:
    for basis in uniqueBasis:
        # Get the CASCI reference energy (scalar)
        casci_row = upDF.loc[
            (upDF["Molecule"] == mol)
            & (upDF["Method"] == refmethod)
            & (upDF["Basis Set"] == basis),
            "Energy",
        ]
        if casci_row.empty:
            continue  # skip if no CASCI reference for this (mol, basis)
        casci_e = casci_row.values[0]  # extract scalar

        # Compute deviations for other methods with the same mol/basis
        mask = (
            (upDF["Molecule"] == mol)
            & (upDF["Basis Set"] == basis)
            # & (upDF["Method"] != refmethod)
        )
        # print(upDF.loc[mask, "Energy"]- casci_e)
        upDF.loc[mask, "Deviation"] = (upDF.loc[mask, "Energy"] - casci_e)*1e3
        # print(upDF.loc[mask, "Energy"] - casci_e)

In [ ]:
upDF.sort_values(by='Energy')

In [ ]:
sns.catplot(upDF,kind='bar',col='Basis Set', x='Molecule', y="Energy", hue='Method')

In [ ]:
#sns.relplot(data=upDF, x="total_bill", y="tip", hue="day", col="time", row="sex")

In [ ]:
# Create the FacetGrid with bar plots
g = sns.catplot(
    data=upDF,
    row="Molecule",
    y="Deviation",
    x="Method",
    hue="Basis Set",
    col="Injected",
    kind="bar",
    height=4,
    aspect=0.6,
)

# Compute global deviation range for consistent shading/scaling
ymin = upDF["Deviation"].min()
ymax = upDF["Deviation"].max()
pad = 0.1 * max(abs(ymin), abs(ymax))

# Apply styling to **all subplots**
for ax in g.axes.flat:
    ax.set_yscale("symlog", linthresh=1e-11)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.axhline(0, color="gray", lw=1, ls="--", alpha=0.8)
    
    # Gray fill region across same y-span
    ax.fill_between(
        np.linspace(-0.5, len(upDF["Method"].unique()) - 0.5, 100),
        -1.6, 1.6,
        color="gray", alpha=0.2
    )

    ax.set_xlim(-0.5, len(upDF["Method"].unique()) - 0.5)

    # Rotate and align x tick labels
    for label in ax.get_xticklabels():
        label.set_rotation(90)
        label.set_horizontalalignment("center")
        label.set_verticalalignment("top")

# Adjust titles and axis labels
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.set_axis_labels("Method", r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")


# Move legend outside the main plot (right side)
g._legend.set_bbox_to_anchor((1.05, 0.5))
g._legend.set_frame_on(False)
plt.subplots_adjust(right=0.85)  # make space for legend

plt.tight_layout()
plt.show()


In [ ]:
g = sns.FacetGrid(upDF, col="Basis Set",row='Molecule')
g.map_dataframe(sns.barplot, x="Method",y='Energy',palette="Paired")

In [ ]:
# # This creates a boolean mask that is True for every row where 'Method' contains "LUCJ"
mask = upDF['Method'].str.contains("LUCJ", case=False, na=False)

# # Use the boolean mask to filter the DataFrame
filtered_df = upDF[mask]


In [ ]:

# devDF = upDF[(upDF['Basis Set']=='STO-3G')&(upDF['Molecule']=='ammonia')]
upDF['Deviation'] = (upDF['Energy'] - upDF.loc[upDF['Method'] == 'CASCI', 'Energy'].values[0])*1e3

In [ ]:
# upDF.loc[(upDF['Molecule']=='formaldehyde')&(upDF['Basis Set']=='STO-3G'),:].sort_values(by=['Method','L','Injected'])

In [ ]:
upDF.sort_values(by=['Method','L','Injected','Deviation'])

In [ ]:
upDF['Method'].value_counts()

In [ ]:
sns.barplot(data=upDF, x='Basis Set', y='Energy', hue='Molecule', errorbar=None)
#plt.xticks(rotation=360-45)
plt.ylabel(r"Energy $\mathrm{kcal\,mol^{-1}}$")
plt.tight_layout()
plt.legend(fontsize=12)
plt.show()

In [ ]:
methods = ['HF', 'SHCI', 'CCSD', 'CASCI']

for method in methods:
    plt.figure(figsize=(8,5))
    subset = upDF[upDF['Method'] == method]
    sns.barplot(data=subset, x='Basis Set', y='Energy', hue='Molecule', errorbar=None)
    plt.title(method)
    #plt.xticks(rotation=360-45)
    
    plt.ylabel(r"Energy $\mathrm{kcal\,mol^{-1}}$")
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    
    plt.show()

In [ ]:
methods = ['HF', 'SHCI', 'CCSD', 'CASCI']

for method in methods:
    plt.figure(figsize=(8,5))
    subset = upDF[upDF['Method'] == method]
    sns.barplot(data=subset, x='Basis Set', y='Deviation', hue='Molecule', errorbar=None)
    plt.title(method)
    #plt.xticks(rotation=360-45)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    
    plt.show()

In [ ]:
heatmap_data = upDF.pivot_table(values='Deviation', index='Molecule', columns='Basis Set', aggfunc='mean') # here aggfunc mean computes the mean deviation
sns.heatmap(heatmap_data, annot=True, cmap='Blues')
plt.title('Mean deviation', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,6))
sns.boxplot(data=upDF, x='Method', y='Deviation', hue='Basis Set')
plt.yscale('symlog')
plt.tight_layout()

plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
sns.stripplot(data=upDF, x='Method', y='Deviation', hue='Basis Set', dodge=True, jitter=True)
plt.yscale('symlog')

plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
plt.tight_layout()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

In [ ]:
for method in methods:
    subset = upDF[upDF['Method'] == method]['Deviation'].dropna()
    stats.probplot(subset, dist="norm", plot=plt)
    plt.title(f"Q–Q plot of deviation ({method})")
    plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
    plt.tight_layout()
    plt.show()

In [ ]:
sns.catplot(data=upDF, x='Basis Set', y='Deviation', hue='Method', col='Molecule', kind='box', col_wrap=3, height=3)
plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
plt.tight_layout()
plt.show()

In [ ]:
for mol in uniqueMol:
    plt.figure(figsize=(9, 5))
    sns.boxplot(
        data=upDF[upDF['Molecule'] == mol],
        x='Method', y='Deviation',
        hue='Basis Set'
    )
    #plt.yscale('symlog')
    plt.title(mol)
    plt.tight_layout() 
    plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
    plt.show()

In [ ]:
n_atoms_dict = {
    'water': 3,
    'methane': 5,
    'ammonia': 4,
    'ethane': 8,
    'methanol': 6,
    'ethylene': 6,
    'formaldehyde': 4
}

upDF['n_atoms'] = upDF['Molecule'].map(n_atoms_dict)

In [ ]:
upDF.head()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=upDF,
    x='n_atoms',
    y='Deviation',
    hue='Method'
)
plt.yscale('symlog')
plt.xlabel("Number of Atoms (Molecule Size)")

plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
plt.tight_layout()
plt.show()


In [ ]:
sns.set_theme(style="whitegrid", context="talk")

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=upDF,
    x='n_atoms',
    y='Deviation',
    hue='Method',
    palette='Set2',
    fliersize=0 
)
sns.stripplot(
    data=upDF,
    x='n_atoms',
    y='Deviation',
    hue='Method',
    dodge=True,
    color='k',
    alpha=0.6,
    size=4
)

plt.yscale('symlog', linthresh=1e-3)
plt.xlabel("Number of Atoms")
plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:len(upDF['Method'].unique())], labels[:len(upDF['Method'].unique())], title="Method", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
upDF

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(
    data=upDF,
    x='Injected',      # categorical injection method
    y='Deviation',
    hue='Method',
    palette='Set2',
    fliersize=0
)
sns.stripplot(
    data=upDF,
    x='Injected',
    y='Deviation',
    hue='Method',
    dodge=True,
    color='k',
    alpha=0.6,
    size=4
)

plt.yscale('symlog', linthresh=1e-3)
plt.xlabel("Injection Method")
plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
plt.title("Energy Deviation by Injection Method and Computational Method")

# Simplify legend
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:len(upDF['Method'].unique())], labels[:len(upDF['Method'].unique())], title="Method", fontsize=12, bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.swarmplot(
    data=upDF,
    x='Method',
    y='Deviation',
    hue='Molecule'
)

plt.ylabel(r"Energy Deviation $\mathrm{kcal\,mol^{-1}}$")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=12)
plt.show()

In [ ]:
sns.pairplot(upDF, hue="Method")